<a href="https://colab.research.google.com/github/Alle84fr/aulas_Kin/blob/main/Algoritmo_palavra_sugerida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

nome: Alessandra Furlanetto Rigonatti 2401151

Uso de biblioteca de IA de pytohn para criar algoritmo de palavra sugerida observando o contexto. Deve utilizar obrigatóriamente as bibliotecas de IA de Python e conceito de probabilidade condicional.
Esta atividade foi pedido no início de abril, porém não foi registrado no Classroom.

In [72]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Pandas = manipular dados
Re = expressões regulares - habilitar biblioteca re para analisar ´padrões de texto

Segundo aula 7 _ probabilidade parte 2

Teoria de Bayes

- evento A (evento que deve ocorrer): palavra que o sistema deve adinhar (dicionário)

- evento B (evento que já ocorreu): palavras que estão sendo digitadas

- p = probabilidade condicional = probabilidade e ocorer a em relação ao evento ocorrido b, ou o evento b, ou ambos

- **p = A/B**

  LLMs - Linguagem de Larga Escala - prevê a p´roxima unidade de texto (token) baseado em todo contexto anterior

- p(A) - vezes que A aparece

- p(B) - vezes que o prefixo, parte de A aparece

- p(A/B) - qual chance de B ser A

- p(B/A) - chance de ser digitado A

- **score (A) = P(A) * P(B/A)**

- n = total de palavras



biblioteca wordfreq =  https://github.com/rspeer/wordfreq

biblioteca difflib = é do Python

bibliopteca sklearn: biblioteca de IA

In [73]:
!pip install wordfreq
!pip install transformers torch

In [74]:
# possui referência a diversas palavras com várias em portugues
from wordfreq import top_n_list, word_frequency

# compara strings
import difflib

# para modelo probabilístico
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

import re
from collections import Counter

# biblioteca .py da Hugging Face, possui modelos já treinados (uso para descobrir palavra sem que eu faça treino por assunto)
from transformers import pipeline

In [75]:
# Criando dicionário
dicionario = top_n_list('pt', 50000)

dicionario_conjunto = set(p for p in dicionario if len(p) >= 2)

qtd_palavras = len(dicionario)

In [76]:
print(f"\nO dicionário possui {qtd_palavras} palavras para serem comparadas")

print(f"\nAlgumas palavras contidas {dicionario_conjunto}\n")


O dicionário possui 50000 palavras para serem comparadas

Algumas palavras contidas {'sis', 'rosto', 'explicação', 'minimizar', 'tecnologia', 'remate', 'passos', 'cola', 'acalma', 'sósia', 'soletrar', 'imersos', 'tônico', 'narrador', 'mejor', 'descansada', 'joffrey', 'conhecíamos', 'caçadas', 'agripino', 'altas', 'deb', 'provam', 'gustave', 'analisa', 'garanti', 'caríssimos', 'alunos', 'recomeçar', 'desfavorecidos', 'gansos', 'retornaremos', 'ae', 'detida', 'substrato', 'esquecer', 'rpc', 'girar', 'regata', 'nele', 'decididamente', 'vingadores', 'considerá', 'acreditaram', 'orquestrada', 'aconselhou', 'store', 'começarei', 'aproximado', 'lab', 'reavaliação', 'proxima', 'matilde', 'buck', 'pelota', 'judgment', 'tce', 'arder', 'trish', 'cibele', 'profusão', 'woolf', 'difere', 'pegado', 'povoados', 'yasser', 'investindo', 'tate', 'palestra', 'theo', 'conversações', 'madonna', 'pentecostais', 'autoria', 'cubs', 'marines', 'irregularmente', 'pus', 'consultor', 'fiques', 'jato', 'antiga', '

In [77]:
def sugerir(digitado, n=3):

  """
  Se a pessoa não digitar, não há o que fazer, se apessoa digitar irá entrar no "else"
  que irá verificar o início da palavras e comparar com possíveis palavras no dict
  Por fim irá retornar uma lista com palavras que deram "match"

  """

  if digitado == "":
        return []

  palavra_sugerida = []

  for palavra in dicionario:

    if palavra.startswith(digitado):
      palavra_sugerida.append(palavra)

  return palavra_sugerida[:n]

In [78]:
def corrigir(palavra, n_sugestoes=5):

  """
  Correção da palavra
  Se a palavra existe, nada acontece, ela está correta
  Se a palavra não existe o Difflib irá analisar a palavra mais "parecida"e com
  70% de match, dar sugestão de uma palavra
  """

  if palavra in dicionario_conjunto:
    return [palavra], True

  sugestoes = difflib.get_close_matches(palavra, dicionario_conjunto, n=n_sugestoes, cutoff=0.7)

  # tratamento das letras que estavam soltas
  sugestoes = [s for s in sugestoes if len(s) >= 3]

  if sugestoes:
    return sugestoes, False

  else:
    return [], False


In [80]:
print("\n\033[1;30;47m=================== 🕵 Provável Palavra 🧑‍🚀 ======================\033[m")
print("                                                                  ")
print("\033[1;30;47m    Digite a inicial de uma para palavra verificar significado    \033[m")
print("                                                                  ")
print("\033[1;30;47m            Digite uma palavra errada para verificar              \033[m")
print("                                                                  ")
print("\033[1;30;47m               Digite 'sair' para encerrar o teste                \033[m")
print("                                                                  ")
print("\033[1;30;47m==================================================================\033[m\n")
print("\033[2;37;40m                Ao digitar aparecerá possíveis palavras           \n                        dê enter para ver a sugest                \033[m\n")

# estava vazando o perto, este obriga a parar
print("\033[0m", end="")

while True:

  digitado = input("Digite: ").lower().strip()

  if digitado == "sair":
    print("Volte quando desejar ")
    break

  if digitado == "":
    print("Sem input.\n")
    continue


  sugestoes = sugerir(digitado)

  if sugestoes:
      print(f"Sugestões para '{digitado}': {sugestoes}")

  else:
        print(f"Nenhuma sugestão encontrada para '{digitado}'")

  if " " not in digitado:
      resultado, correta = corrigir(digitado)

      if correta:
          print(f"🥳 '{digitado},' acertei?")

      elif sugestoes:
          print(f"🧐 Você quis dizer: '{sugestoes}'?")

      else:
           print(f"😶 Palavra não reconhecida")

  print()



=================== 🕵 Provável Palavra 🧑‍🚀 ======================
                                                                  
    Digite a inicial de uma para palavra verificar significado    
                                                                  
            Digite uma palavra errada para verificar              
                                                                  
               Digite 'sair' para encerrar o teste                
                                                                  

                Ao digitar aparecerá possíveis palavras           
                        dê enter para ver a sugest                

Digite: pi
Sugestões para 'pi': ['pior', 'pista', 'piada']
🥳 'pi,' acertei?

Digite: pir
Sugestões para 'pir': ['pires', 'pirata', 'piratas']
🧐 Você quis dizer: '['pires', 'pirata', 'piratas']'?

Digite: piru
Sugestões para 'piru': ['pirulito']
🧐 Você quis dizer: '['pirulito']'?

Digite: ab
Sugestões para 'ab': ['abril', 'abai

**difflib**

Pega o  lens da palavra

Analisa a similaridade que (2*letras em comum)/palavra Input + palavra provavel

ex:

palavra input =ornit -> len = 5

palavra provavel = ornitorrinco -> len = 12

letras em comum = ornit = 5

*similaridade* = (2*5)/(5+12) = 10/17 = 0.58 (58%)*

Para que o cutoff ( cutoff=0.7) deve ter similaridade >= a 0.70 (70%)

O mesmo ocorre quando há palavras erradas


In [81]:
gerador = pipeline('fill-mask', model='neuralmind/bert-base-portuguese-cased')

def analisar_texto(texto):
    """
    Recebe texto digitado pelo usuário, calcula % de acerto/erro, lista palavras
    e prevê próxima palavra observando o CONTEXTO com modelo pré-treinado em português
    """

    palavras = re.sub(r'[^a-záéíóúãõâêîôûàç\s]', '', texto.lower()).split()

    if not palavras:
        print("Nenhuma palavra encontrada.")
        return

    total = len(palavras)
    erros = []
    certas = 0

    #___________ verifica cada palavra

    for palavra in palavras:
        resultado, correta = corrigir(palavra)

        if correta:
            certas += 1

        else:
            erros.append((palavra, resultado))

    pct_acerto = round((certas / total) * 100)
    pct_erro = 100 - pct_acerto

    #___________ resultado geral

    print("\n========== Resultado ==========\n")

    if pct_acerto >= 60:
        print(f"🤩 Acertou {pct_acerto}% das palavras")

    else:
        print(f"😐 Errou {pct_erro}% das palavras")

    #___________ palavras erradas com até 5 sugestões

    if erros:
        print("\n🔴 Palavras com erro e possíveis correções:")

        for palavra, sugestoes in erros:

            if sugestoes:
                lista = ", ".join(
                    f"{i+1}.{s}" for i, s in enumerate(sugestoes)
                )
                print(f"✍ {palavra} → {lista}")

            else:
                print(f"⛑️ {palavra} → não reconhecida")

    else:
        print("\n🪹 Nenhum erro encontrado")

    #___________ próxima palavra

    # [MASK] marca o ponto onde irá prever a próxima palavra
    frase_com_mask = texto.replace("?", " [MASK]")

    try:
        # vinte sugestções para depois tirar as 5 "melhores"
        resultado = gerador(frase_com_mask, top_k=20)

        # primeira rodada apareceu : e ., então fiz a lista negra
        black_list = {
            '.', ',', ':', ';', '!', '?', '-', '_',
            '(', ')', '[', ']', '"', "'", '/',
            'de', 'a', 'o', 'e', 'que', 'em',
            'um', 'uma', 'os', 'as', 'do', 'da',
            'no', 'na', 'se', 'por', 'com', 'para'
        }

        filtradas = [
            r for r in resultado
            if r['token_str'].strip() not in black_list
            # .isalpha() → apenas letras
            and r['token_str'].strip().isalpha()
        ]

        print("\n🔮 Talvez a próxima palavra seja...")

        for r in filtradas[:5]:
            palavra = r['token_str'].strip()
            pct = round(r['score'] * 100, 1)
            print(f" {palavra:15} {pct}%")

    except Exception:
        print("\n🥷 Não foi possível prever a palavra")

    print("===============================\n")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [82]:
print("\033[1;30;47m  Digite um ponto de interrogação  - ? - no lugar da palvra a ser adivinhada  \033[m")
print("                  Exemplo: Hoje fui à praia e vi ? no mar                     ")
texto_usuario = input("\nDigite seu texto: ")
print("\033[0m", end="")

analisar_texto(texto_usuario)

  Digite um ponto de interrogação  - ? - no lugar da palvra a ser adivinhada  
                  Exemplo: Hoje fui à praia e vi ? no mar                     

Digite seu texto: Hoje comerei um plato de ? na janta

========== Resultado ==========

🤩 Acertou 71% das palavras

🔴 Palavras com erro e possíveis correções:
✍ comerei → 1.começarei, 2.comprei, 3.começei, 4.comeria, 5.comerem
✍ plato → 1.platão, 2.palato, 3.pato, 4.lato, 5.plantão

🔮 Talvez a próxima palavra seja...
 carne           17.5%
 arroz           8.4%
 queijo          8.1%
 peixe           7.0%
 feijão          4.6%



Vi que para funcionar como queeria, co a porcentagem e sugestão mais "afiada", deveria usar o GPT e não o Bert.

O GPT = Generative Pre-traines Transfrome lê a frase da e prevê o que pode vir depois, o marck seria o lado direito. Ela gera texto que continuam a frase, mantendo o sentido.

O Bert = Bidirectional Encoder Representations from Transformers, lê a frase da em todos os sentidos, ele entende o contexto e preenche lacunas entre as palvras